In [1]:
import warnings

warnings.filterwarnings("ignore")


from datetime import date, timedelta
from pypfopt import expected_returns
import pyfolio as pf
import ffn
import qgrid

import datetime
from empyrical.stats import  *

# 
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from pypfopt.expected_returns import returns_from_prices

import quandl
quandl.ApiConfig.api_key = 'fhbmNKX6oNP7PpFuZJNo'

# for downloading data 
from fredapi import Fred

import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

In [2]:
cd /Users/safishajjouz/GitHub/myPythonPackages

/Users/safishajjouz/GitHub/myPythonPackages


In [3]:
from myPortfolioManagement.myData import * 
from myPortfolioManagement.myPerformanceAnalytics import *

In [4]:
# import stock prices 
df_sp_stock_price = pd.read_csv('/Users/safishajjouz/Dropbox/Schonfeld_shared_folder/sp_prices.csv')

In [5]:
# import inflation 
# Set dates 
start_date = '1993-01-01'
date_end = date.today()
date_end = date_end.strftime("%Y-%m-%d")

df_bench = get_stock_prices(yahoo_tickers = ['^GSPC'], 
                 start_date = start_date,
                 end_date = date_end, 
                 time_interval = 'daily', 
                 num_cpus = 5) 

get_stock_prices took 5.070sec


In [6]:
df_sp_stock_price = df_sp_stock_price.pivot(index = 'ref.date', columns='ticker', 
                    values='price.adjusted')

In [7]:
df_sp_stock_price = df_sp_stock_price.reset_index().rename(columns = {'ref.date':'Date'}).set_index('Date')

In [8]:
ret = returns_from_prices(df_sp_stock_price)

In [9]:
ret.index = pd.to_datetime(ret.index)
ret  = ret.resample('M').mean()

In [10]:
ret['year'] = pd.DatetimeIndex(ret.index).year
ret['month'] = pd.DatetimeIndex(ret.index).month

In [11]:
my_fred_API = 'cc628b51e21828ae6b98c06f4eef6714' # 
fred = Fred(api_key=my_fred_API)

series_to_download = ['CPIAUCSL', # CPI all items #  
                     'USACPICORMINMEI'] #CPI ex-food and energy

# choose frequency 
freq = ['m', 'q', 'a'][0]

df = {}
for series_id in series_to_download:
    info = fred.get_series_info(series_id)['title']
    print(info)
    df[series_id] = fred.get_series(series_id, frequency = freq)
df = pd.DataFrame(df)
df = df.rename(columns = {'CPIAUCSL':'CPI', 'USACPICORMINMEI':'CPI_core' })

Consumer Price Index for All Urban Consumers: All Items in U.S. City Average
Consumer Price Index: All Items Excluding Food and Energy for the United States


In [12]:
df = df [ df.index>= start_date] 

In [13]:
df = df.reset_index().rename(columns = {'index':'Date'}).set_index('Date')

In [14]:
df_inf = returns_from_prices(df)

In [15]:
df_inf.index = pd.to_datetime(df_inf.index)

In [16]:
df_inf['year'] = pd.DatetimeIndex(df_inf.index ).year
df_inf['month'] = pd.DatetimeIndex(df_inf.index ).month

In [17]:
ret = ret.merge(df_inf, on = ['year', 'month'])

In [18]:
df_greeks = alpha_beta(ret.drop(['year', 'month'], axis = 1), benchmark = 'CPI', 
                       my_date_col_name = 'Date', 
                       returns_rolling = False)

In [19]:
df_greeks.sort_values('beta', ascending = False)

,alpha,beta
fund,,
NVR,0.95,1.07
PXD,-0.03,0.36
FCX,0.04,0.30
QCOM,0.17,0.29
HAL,0.02,0.29
...,...,...
CTXS,0.49,-0.27
ALK,0.42,-0.35
UAL,0.36,-0.37


In [21]:
import pandas as pd 
pd.set_option('display.float_format', lambda x: '%.3f' % x)
from finvizfinance.screener.overview import Overview

import plotly.express as px

In [27]:
# for filtering: https://finviz.com/screener.ashx
foverview = Overview()
filters_dict = {'Index': 'S&P 500'}
foverview.set_filter(filters_dict=filters_dict)
df_stock_info = foverview.ScreenerView()

In [26]:
df_greeks = df_greeks.reset_index().rename(columns = {'fund': 'Ticker'})

In [31]:
df_stock_info = df_stock_info[['Ticker', 'Company', 'Sector']].merge(df_greeks, on = 'Ticker')

In [37]:
df_stock_info = df_stock_info.rename(columns = {'beta':'inflation_beta'})

In [40]:
df_stock_info.to_csv('/Users/safishajjouz/GitHub/myPortfolioManagement/stock_inf_sen.csv', index=False)